## Table VI: CRTAT Component Ablation (replay, same protocol as Table IV)

Per-attack AUC (positives = that attack type, negatives = 25 honest clients), pooled AUC, and F1 / #flagged at the two labelled operating points; mean ± std over `SEEDS`. The text comparing variants is generated from the table (see the printed 'AUTO-GENERATED ABLATION STATEMENT').


In [ ]:
# ─── Table VI: CRTAT component ablation (replay; same protocol as the main run) ───────────────
# Per-attack AUC: positives = that attack type, negatives = the 25 honest clients, per seed → mean±std.
# F1 / #flagged are shown at the two labelled operating points (OP-A principal, OP-B secondary).
def table_vi(ab, dname):
    print(f'\nTable VI — CRTAT component ablation | {dname} | replay of recorded signals | mean±std over seeds {SEEDS}')
    print('═'*128)
    print(f'{"Variant":<44}{"Overt AUC":>13}{"Slow AUC":>13}{"Inv. AUC":>13}{"Pooled AUC":>13}{"F1 OP-A":>13}{"F1 OP-B":>13}{"#flag OP-A":>10}')
    print('─'*128)
    for sep, rows in [(False, ABLATION_VARIANTS), (True, EXTENSION_VARIANTS)]:
        if sep: print('  · extensions added after review ·')
        for name, _ in rows:
            m, s = ab[name]['mean'], ab[name]['std']
            cells = [f'{m[k]:.3f}±{s[k]:.3f}' for k in range(6)] + [f'{m[6]:.1f}']
            print(f'{name:<44}' + ''.join(f'{c:>{w}}' for c, w in zip(cells, [13,13,13,13,13,13,10])))
    print('═'*128)

table_vi(abl_cifar, 'CIFAR-10')
table_vi(abl_mnist, 'MNIST')

# ── Auto-generated comparison text (reviewer §3B) — the sentence is derived from the table, never typed by hand ──
def ablation_statement(ab, dname, a='Instantaneous-only', b='Full CRTAT'):
    names = ['overt-attack', 'slow-attack', 'inversion']; better, worse, same = [], [], []
    for k, nm in enumerate(names):
        d = ab[a]['mean'][k] - ab[b]['mean'][k]
        (better if d > 1e-3 else worse if d < -1e-3 else same).append(f'{nm} ({ab[a]["mean"][k]:.3f} vs {ab[b]["mean"][k]:.3f})')
    parts = []
    if better: parts.append(f'higher {a} AUC than {b} for ' + ', '.join(better))
    if worse:  parts.append(f'lower for ' + ', '.join(worse))
    if same:   parts.append(f'equal (±0.001) for ' + ', '.join(same))
    return f'{dname}: ' + '; '.join(parts) + '.'
print('\nAUTO-GENERATED ABLATION STATEMENT (paste into the manuscript instead of hand-written comparisons):')
print('  ' + ablation_statement(abl_cifar, 'CIFAR-10'))
print('  ' + ablation_statement(abl_mnist, 'MNIST'))
print('  (the previous manuscript text claimed Instantaneous-only had higher overt AND slow AUC on CIFAR-10;')
print('   check the generated sentence above against the rerun table before reusing any wording)')

# Components whose removal does not hurt (honest reporting — these motivate refinement, not a blanket claim)
print('\nComponents whose removal INCREASES pooled AUC (Δ>0 vs Full) — report these as findings:')
for dname, ab in [('CIFAR-10', abl_cifar), ('MNIST', abl_mnist)]:
    fullp = ab['Full CRTAT']['mean'][3]
    hits = [(n, ab[n]['mean'][3]-fullp) for n, _ in ABLATION_VARIANTS if n != 'Full CRTAT' and ab[n]['mean'][3] - fullp > 1e-3]
    print(f'  {dname}: ' + (', '.join(f'{n} ({d:+.3f})' for n, d in hits) if hits else 'none'))
# 'w/o Inversion' identical to Full? check whether the inversion term changes ANY rank
for dname, results in [('CIFAR-10', results_cifar), ('MNIST', results_mnist)]:
    same_rank = all(np.array_equal(np.argsort(crtat_replay(r)[0], kind='stable'),
                                   np.argsort(crtat_replay(r, use_inversion=False)[0], kind='stable')) for r in results)
    maxdiff = max(np.abs(crtat_replay(r)[0] - crtat_replay(r, use_inversion=False)[0]).max() for r in results)
    print(f'  {dname}: inversion term changes client ranking? {"NO (AUC identical by construction)" if same_rank else "yes"}  | max |Δscore| = {maxdiff:.4f}')



Table VI — CRTAT component ablation | CIFAR-10 | replay of recorded signals | mean±std over seeds [42, 123]
════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Variant                                         Overt AUC     Slow AUC     Inv. AUC   Pooled AUC      F1 OP-A      F1 OP-B#flag OP-A
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Instantaneous-only                            0.660±0.220  0.440±0.067  1.000±0.000  0.596±0.004  0.286±0.000  0.357±0.071      30.0
w/o Trend                                     0.460±0.060  0.380±0.060  0.980±0.020  0.516±0.020  0.215±0.015  0.214±0.071      18.0
w/o Acceleration                              0.500±0.180  0.427±0.133  0.980±0.020  0.552±0.040  0.244±0.022  0.214±0.071      23.5
w/o Volatility                                0.540±0.140  0.387±0.053  1.000±0.000  0.540±0.060  0.2